# SadTalker — Generador de Avatar
Ejecuta este notebook en Google Colab con GPU activada.
Genera un vídeo con avatar hablante a partir de audio + imagen.

In [ ]:
# PASO 0 — Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Carpeta de tu proyecto en Drive — cambia esta ruta si es necesario
CARPETA_DRIVE = '/content/drive/MyDrive/Proyecto_DeepLearning'
print('Drive montado!')

In [ ]:
# PASO 1 — Instalar SadTalker
!git clone https://github.com/OpenTalker/SadTalker.git
%cd SadTalker
!pip install -r requirements.txt

In [ ]:
# PASO 2 — Descargar modelos preentrenados
!bash scripts/download_models.sh

In [ ]:
# PASO 3 — Copiar archivos desde Drive a Colab
import os, shutil, glob

# Cogemos el audio más reciente de la carpeta audios/
audios = sorted(glob.glob(f'{CARPETA_DRIVE}/audios/audio_*.mp3'))
if not audios:
    raise Exception('No se encontró ningún audio en Drive. Asegúrate de que está en Proyecto_DeepLearning/audios/')
ruta_audio = audios[-1]  # el más reciente

# Cogemos la imagen del avatar
ruta_imagen = f'{CARPETA_DRIVE}/avatares/avatar.jpg'
if not os.path.exists(ruta_imagen):
    raise Exception('No se encontró avatar.jpg en Drive. Asegúrate de que está en Proyecto_DeepLearning/avatares/')

# Copiamos a Colab
shutil.copy(ruta_audio,  '/content/audio.mp3')
shutil.copy(ruta_imagen, '/content/avatar.jpg')

print(f'Audio:  {ruta_audio}')
print(f'Imagen: {ruta_imagen}')
print('Archivos copiados a Colab!')

In [ ]:
# PASO 4 — Generar vídeo con SadTalker
!python inference.py \
    --driven_audio /content/audio.mp3 \
    --source_image /content/avatar.jpg \
    --result_dir /content/resultado \
    --still \
    --preprocess full \
    --enhancer gfpgan

In [ ]:
# PASO 5 — Guardar vídeo en Drive
import glob, shutil
from datetime import datetime

# Buscamos el vídeo generado
videos = glob.glob('/content/resultado/*.mp4')
if not videos:
    raise Exception('No se generó ningún vídeo. Revisa el paso anterior.')

# Lo guardamos en Drive con timestamp
timestamp    = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
carpeta_dest = f'{CARPETA_DRIVE}/videos'
os.makedirs(carpeta_dest, exist_ok=True)
destino = f'{carpeta_dest}/avatar_sin_subs_{timestamp}.mp4'

shutil.copy(videos[-1], destino)
print(f'Video guardado en Drive: {destino}')